# A neobank support agent — governed tools on a Snowflake MCP server

- Trusted support-ops agent — **not** a public chatbot
- Authenticates as a locked-down `CUSTOMER_AGENT` role → its whole world is the 4 tools below
- Each tool = a real Snowflake object, exposed through **one** MCP server

| Tool | Snowflake object | Type |
|---|---|---|
| `classify_intent` | `CLASSIFY_INTENT_PROC` → fine-tuned Llama 3.1 8B | Stored procedure |
| `search_help_articles` | `SEARCH_HELP_ARTICLES` | Cortex Search |
| `get_transaction_status` | `GET_TRANSACTION_STATUS` | SQL UDF |
| `file_ticket` | `FILE_TICKET` → `TICKETS` (routes by intent) | Stored procedure |

**`classify_intent`** is the star — a decision boundary *you own*: triages into 1 of **77 Banking77 intents** and beats frontier models even when they get the full label list. That's why a fine-tune is *necessary*, not just cheaper.

## 1. The data

In [ ]:
-- Help-centre knowledge base (what search_help_articles searches)
SELECT ARTICLE_ID, TITLE, CATEGORY, LEFT(BODY, 90) || '...' AS BODY_PREVIEW
FROM MCP_HOL.SUPPORT.HELP_ARTICLES
ORDER BY ARTICLE_ID

In [ ]:
-- Support cases / transactions (what get_transaction_status looks up)
SELECT * FROM MCP_HOL.SUPPORT.CASES ORDER BY REF_ID

In [ ]:
-- Incident log (what file_ticket writes to)
SELECT * FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC

In [ ]:
-- Routing policy: intent -> queue / priority / SLA (all 77 intents)
SELECT QUEUE, COUNT(*) AS intents, MIN(PRIORITY) AS top_priority
FROM MCP_HOL.SUPPORT.INTENT_ROUTING
GROUP BY QUEUE ORDER BY intents DESC

## 2. Tool 1 — `classify_intent` (the fine-tune)

- **Stored procedure** over an in-account **fine-tuned Llama 3.1 8B** (Banking77: 77 intents, 10k labeled messages)
- In = the **raw customer message** (no instructions, no label list) → Out = one label, e.g. `card_arrival`
- The task + 77 labels live in the model's **weights**

**Why a fine-tune, not a prompt?** Telling `card_arrival` from `lost_or_stolen_card` is a convention learned from thousands of tickets — not a promptable rule. (Proof below.)

**Why a procedure, not a UDF?** A fine-tuned model is reachable through the MCP server only from an owner's-rights session (`EXECUTE AS OWNER`); the GENERIC function path can't resolve it.

Trained once, offline (kept out of Run All):
```sql
SELECT SNOWFLAKE.CORTEX.FINETUNE('CREATE', 'MCP_HOL.SUPPORT.SUPPORT_INTENT_8B', 'llama3.1-8b',
  $$SELECT TEXT AS prompt, LABEL AS completion FROM MCP_HOL.SUPPORT.B77_TRAIN$$,
  $$SELECT TEXT AS prompt, LABEL AS completion FROM MCP_HOL.SUPPORT.B77_PROBE$$);
```

In [ ]:
-- The MCP tool wrapper: a stored procedure so the fine-tuned model is reachable
-- through the managed MCP server. Bare message in -> one Banking77 label out.
CREATE OR REPLACE PROCEDURE MCP_HOL.SUPPORT.CLASSIFY_INTENT_PROC(MESSAGE STRING)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = 3.11
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
EXECUTE AS OWNER
AS $$
def run(session, message):
    row = session.sql(
        "SELECT SNOWFLAKE.CORTEX.COMPLETE('MCP_HOL.SUPPORT.SUPPORT_INTENT_8B', ?)",
        params=[message or '']).collect()
    raw = (row[0][0] or '').strip().lower()
    import re
    m = re.search(r'[a-z_]{3,}', raw)
    return m.group(0) if m else raw
$$

In [ ]:
-- input: a customer message   ->   output: one banking intent label
CALL MCP_HOL.SUPPORT.CLASSIFY_INTENT_PROC('My new debit card still has not arrived after two weeks.')

## The proof: fine-tune vs frontier

- Held-out **154-message probe**, exact-match scoring
- Fine-tune gets **only the bare message**; `claude-4-sonnet` + `openai-gpt-4.1` get the **full 77-label list** (their best honest setup)
- Fine-tune wins by **~12 points** — the boundary is tacit, not promptable
- (Runs ~1–2 min: 154 × 3 models)

In [ ]:
-- Fair benchmark: FT (bare message) vs frontier (full label list). Exact-match.
WITH g AS (
  SELECT LISTAGG(DISTINCT LABEL, ', ') WITHIN GROUP (ORDER BY LABEL) AS labels
  FROM MCP_HOL.SUPPORT.B77_TRAIN
),
base AS (
  SELECT
    LOWER(TRIM(p.LABEL)) AS truth,
    LOWER(TRIM(SNOWFLAKE.CORTEX.COMPLETE('MCP_HOL.SUPPORT.SUPPORT_INTENT_8B', p.TEXT))) AS ft_raw,
    LOWER(TRIM(AI_COMPLETE('claude-4-sonnet',
      'You are an intent classifier for a neobank. Classify the message into exactly ONE of these 77 intents. Reply with ONLY the label (lowercase, underscores).'
      || CHR(10) || 'Intents: ' || g.labels || CHR(10) || 'Message: ' || p.TEXT))) AS sonnet_raw,
    LOWER(TRIM(AI_COMPLETE('openai-gpt-4.1',
      'You are an intent classifier for a neobank. Classify the message into exactly ONE of these 77 intents. Reply with ONLY the label (lowercase, underscores).'
      || CHR(10) || 'Intents: ' || g.labels || CHR(10) || 'Message: ' || p.TEXT))) AS gpt_raw
  FROM MCP_HOL.SUPPORT.B77_PROBE p, g
),
norm AS (
  SELECT truth,
    REGEXP_SUBSTR(ft_raw,     '[a-z_]{3,}', 1, GREATEST(REGEXP_COUNT(ft_raw,     '[a-z_]{3,}'),1)) AS ft,
    REGEXP_SUBSTR(sonnet_raw, '[a-z_]{3,}', 1, GREATEST(REGEXP_COUNT(sonnet_raw, '[a-z_]{3,}'),1)) AS sonnet,
    REGEXP_SUBSTR(gpt_raw,    '[a-z_]{3,}', 1, GREATEST(REGEXP_COUNT(gpt_raw,    '[a-z_]{3,}'),1)) AS gpt
  FROM base
)
SELECT 'fine-tuned llama3.1-8b (bare message)' AS model, ROUND(AVG(IFF(ft=truth,1,0))*100,1) AS accuracy_pct FROM norm
UNION ALL SELECT 'openai-gpt-4.1 (full 77-label list)', ROUND(AVG(IFF(gpt=truth,1,0))*100,1) FROM norm
UNION ALL SELECT 'claude-4-sonnet (full 77-label list)', ROUND(AVG(IFF(sonnet=truth,1,0))*100,1) FROM norm
ORDER BY accuracy_pct DESC

## 3. Tool 2 — `search_help_articles`

- Cortex Search over the bank's help-centre `HELP_ARTICLES`
- Agent sends a natural-language query → gets the most relevant articles to ground its answer
- Auto-generated input schema: only `query` required (`columns`/`filter`/`limit` optional)
- **Column names live in the tool description** — the only place the agent learns them

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE MCP_HOL.SUPPORT.SEARCH_HELP_ARTICLES
  ON BODY
  ATTRIBUTES ARTICLE_ID, TITLE, CATEGORY
  WAREHOUSE = COCO_WH
  TARGET_LAG = '1 hour'
  AS SELECT BODY, ARTICLE_ID, TITLE, CATEGORY
     FROM MCP_HOL.SUPPORT.HELP_ARTICLES

In [ ]:
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'MCP_HOL.SUPPORT.SEARCH_HELP_ARTICLES',
  '{"query":"how long does a new card take to arrive","limit":2}'
) AS output

## 4. Tool 3 — `get_transaction_status`

- SQL UDF returning one case/transaction's status by `ref_id` (e.g. `CASE-10001`)
- **Owner's rights** — the caller runs it without `SELECT` on the `CASES` table
- Out: one line, e.g. `Case CASE-10001 (New debit card delivery): Dispatched, expected by 2026-02-12 ...`

In [ ]:
CREATE OR REPLACE FUNCTION MCP_HOL.SUPPORT.GET_TRANSACTION_STATUS(REF_ID VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
WITH input AS (SELECT REF_ID AS rid)
SELECT COALESCE(
  (SELECT 'Case ' || c.REF_ID || ' (' || c.TOPIC || '): ' || c.STATUS ||
          CASE WHEN c.STATUS IN ('Dispatched','In Progress','Pending')
               THEN ', expected by ' || TO_VARCHAR(c.ETA, 'YYYY-MM-DD') ELSE '' END ||
          CASE WHEN c.DETAIL IS NOT NULL THEN ' (' || c.DETAIL || ')' ELSE '' END || '.'
   FROM MCP_HOL.SUPPORT.CASES c, input i
   WHERE c.REF_ID = i.rid),
  'No case found for ' || (SELECT rid FROM input) || '.')
$$

In [ ]:
-- input: a case reference   ->   output: one line for that case
SELECT MCP_HOL.SUPPORT.GET_TRANSACTION_STATUS('CASE-10001') AS output

## 5. Tool 4 — `file_ticket`

- Stored procedure: files an incident to `TICKETS`, **routed by the `intent` label** from `classify_intent`
- Looks up `INTENT_ROUTING` → sets queue / priority / SLA (the model's output decides where it goes)
- Self-contained, owner's rights
- In: `ref_id`, `issue`, `intent` → Out: `Created incident INC0001001 ... routed to Cards / P3 (SLA 24h).`

In [ ]:
CREATE OR REPLACE PROCEDURE MCP_HOL.SUPPORT.FILE_TICKET(REF_ID STRING, ISSUE STRING, INTENT STRING)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = 3.11
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
EXECUTE AS OWNER
AS $$
def run(session, ref_id, issue, intent):
    label = (intent or '').strip().lower()
    row = session.sql(
        'SELECT QUEUE, PRIORITY, SLA_HOURS FROM MCP_HOL.SUPPORT.INTENT_ROUTING WHERE INTENT = ?',
        params=[label]).collect()
    if row:
        queue, priority, sla = row[0][0], row[0][1], row[0][2]
    else:
        label, queue, priority, sla = 'unclassified', 'General Support', 'P4', 48
    seq = session.sql('SELECT MCP_HOL.SUPPORT.TICKET_SEQ.NEXTVAL').collect()[0][0]
    inc = 'INC' + str(seq).zfill(7)
    session.sql(
        'INSERT INTO MCP_HOL.SUPPORT.TICKETS '
        '(INCIDENT_NUMBER, REF_ID, ISSUE, INTENT, QUEUE, PRIORITY, STATUS, CREATED_AT) '
        "VALUES (?, ?, ?, ?, ?, ?, 'New', CURRENT_TIMESTAMP())",
        params=[inc, ref_id, issue, label, queue, priority]).collect()
    return ('Created incident ' + inc + ' for case ' + str(ref_id)
            + ' — classified as ' + label + ', routed to the ' + queue
            + ' queue at priority ' + priority + ' (SLA ' + str(sla) + 'h).')
$$

In [ ]:
-- input: case ref + issue + intent (from classify_intent)   ->   routed incident
CALL MCP_HOL.SUPPORT.FILE_TICKET('CASE-10001', 'New debit card has not arrived after two weeks', 'card_arrival')

## 6. Create the MCP server

One statement exposes the four objects above as agent tools for the external Gemini agent.

In [ ]:
CREATE OR REPLACE MCP SERVER MCP_HOL.AGENTS.MCP_HOL
FROM SPECIFICATION $$
tools:
  - name: "classify_intent"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.CLASSIFY_INTENT_PROC"
    title: "Classify customer intent"
    description: "Classify a customer's message into exactly one of 77 neobank support intents (e.g. card_arrival, lost_or_stolen_card, declined_card_payment, exchange_rate, verify_my_identity) using an in-account FINE-TUNED model. Returns one lowercase_with_underscores label. Call this FIRST to triage the message; its label is REQUIRED by file_ticket to route the case."
    config:
      type: "procedure"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          message: { type: "string", description: "The customer's message to classify" }
        required: ["message"]
  - name: "search_help_articles"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "MCP_HOL.SUPPORT.SEARCH_HELP_ARTICLES"
    title: "Search help-centre articles"
    description: "Search the bank's help-centre knowledge base for a topic (card delivery times, lost/stolen card, declined payments, transfers, exchange rates, fees, identity verification) and return the most relevant articles. Each article has BODY plus attributes ARTICLE_ID, TITLE, CATEGORY; pass any of those names in the optional 'columns' arg (only BODY is returned by default), and filter on CATEGORY via 'filter'."
  - name: "get_transaction_status"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.GET_TRANSACTION_STATUS"
    title: "Look up a case or transaction"
    description: "Look up the status of a SINGLE support case or transaction by its reference id. Returns the topic, current status, expected date, and channel detail."
    config:
      type: "function"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          ref_id: { type: "string", description: "The case/transaction reference, e.g. CASE-10001" }
        required: ["ref_id"]
  - name: "file_ticket"
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.FILE_TICKET"
    title: "Open a support incident"
    description: "File a support incident for the customer's case. Requires the intent label from classify_intent, which routes the incident to the right queue/priority. Returns the incident number and routing."
    config:
      type: "procedure"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          ref_id: { type: "string", description: "The related case reference, e.g. CASE-10001" }
          issue: { type: "string", description: "Short description of the customer's problem" }
          intent: { type: "string", description: "The intent label from classify_intent (e.g. card_arrival). Used to route the incident." }
        required: ["ref_id", "issue", "intent"]
$$;

## 7. Secure & authenticate the agent

Creating the server isn't the security story — **how the agent connects** is. Two gates, neither relying on the agent behaving:

1. **What tools exist** — exactly these 4. No `SYSTEM_EXECUTE_SQL`, no financials, no way to move money.
2. **What the role can touch** — connects as `CUSTOMER_AGENT`; one `DEFAULT_ROLE`, **no secondary roles**. Hard ceiling.

**This demo's shape:** the customer chats with *your app*; your app holds one credential and connects as `CUSTOMER_AGENT`. The customer never authenticates to Snowflake.

In [ ]:
-- Least-privilege: USAGE on the server + each tool, nothing else (owner's rights = no SELECT needed).
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS CUSTOMER_AGENT;

GRANT USAGE ON DATABASE MCP_HOL                 TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON SCHEMA MCP_HOL.SUPPORT           TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON SCHEMA MCP_HOL.AGENTS            TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON WAREHOUSE COCO_WH                TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON MCP SERVER MCP_HOL.AGENTS.MCP_HOL TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON PROCEDURE MCP_HOL.SUPPORT.CLASSIFY_INTENT_PROC(VARCHAR)          TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON FUNCTION  MCP_HOL.SUPPORT.GET_TRANSACTION_STATUS(VARCHAR)        TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON PROCEDURE MCP_HOL.SUPPORT.FILE_TICKET(VARCHAR, VARCHAR, VARCHAR) TO ROLE CUSTOMER_AGENT;
GRANT USAGE ON CORTEX SEARCH SERVICE MCP_HOL.SUPPORT.SEARCH_HELP_ARTICLES       TO ROLE CUSTOMER_AGENT;

-- Point your agent's service user at the scoped role (replace <agent_user>).
GRANT ROLE CUSTOMER_AGENT TO USER <agent_user>;
ALTER USER <agent_user> SET DEFAULT_ROLE = 'CUSTOMER_AGENT' DEFAULT_WAREHOUSE = 'COCO_WH';

### Authenticate — one service token

- **This demo → role-restricted PAT** (what our Gemini agent uses): your app's one token, capped at `CUSTOMER_AGENT` no matter the user's other roles
- **Per-human clients** (Claude/Cursor/ChatGPT) → OAuth instead: each person signs in, session = their `DEFAULT_ROLE`

In [ ]:
-- This demo: role-restricted PAT for the app's service user (replace <agent_user>).
ALTER USER <agent_user> ADD PROGRAMMATIC ACCESS TOKEN customer_agent_pat
  ROLE_RESTRICTION = 'CUSTOMER_AGENT'
  DAYS_TO_EXPIRY = 7;

-- Per-human client alternative (Claude/Cursor): OAuth instead of a PAT.
CREATE OR REPLACE SECURITY INTEGRATION MCP_OAUTH
  TYPE = OAUTH
  OAUTH_CLIENT = CUSTOM
  ENABLED = TRUE
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = '<your_mcp_client_redirect_uri>';

### Restrict where connections come from (network policy)

- Admit only your app's published egress IPs
- This demo (PAT): attach the policy to the **service user** — that's what gates its token (for OAuth clients, attach to the integration instead)

In [ ]:
-- Allow only the app's published egress IPs.
CREATE OR REPLACE NETWORK RULE MCP_HOL.AGENTS.MCP_CLIENT_INGRESS
  MODE = INGRESS
  TYPE = IPV4
  VALUE_LIST = ('<agent_egress_ip_1>', '<agent_egress_ip_2>');

CREATE NETWORK POLICY IF NOT EXISTS MCP_POLICY
  ALLOWED_NETWORK_RULE_LIST = ('MCP_HOL.AGENTS.MCP_CLIENT_INGRESS');

-- This demo (PAT): gate the service user.
ALTER USER <agent_user> SET NETWORK_POLICY = MCP_POLICY;
-- OAuth-client alternative: gate the integration instead.
-- ALTER SECURITY INTEGRATION MCP_OAUTH SET NETWORK_POLICY = MCP_POLICY;

### Prove the scope holds

- As `CUSTOMER_AGENT` (no secondary roles): the tool still works (owner's rights)...
- ...but reading the raw table (`SELECT * FROM CASES`) is **denied**
- The role caps *what* the agent can do; *whose* data it sees is your app's job (it passes the customer's id)

In [ ]:
USE ROLE CUSTOMER_AGENT;
USE SECONDARY ROLES NONE;

-- In scope: the tool works without SELECT on the underlying table (owner's rights).
SELECT MCP_HOL.SUPPORT.GET_TRANSACTION_STATUS('CASE-10001') AS in_scope_tool_works;

In [ ]:
-- Out of scope: no SELECT on the table -> access denied (expected).
SELECT * FROM MCP_HOL.SUPPORT.CASES;

In [ ]:
-- Restore your working context after the scope test.
USE SECONDARY ROLES ALL;
USE ROLE ACCOUNTADMIN;